# 一维梯度下降拟合线性回归 · 交互式演示

本 Notebook 用**二维参数空间 (w, b) 上的梯度下降**，演示如何从零拟合一个线性回归模型。

**流程**
1. 生成数据：一次函数 + 正态噪声
2. 定义平方误差损失 $J(w,b)$，推导对 $w,b$ 的偏导
3. 用固定学习率迭代更新 $w,b$
4. 交互演示：左图看 $(w,b)$ 在损失曲面上的下降轨迹（带箭头），右图看对应的函数拟合效果
5. 对比不同学习率
6. 训练全过程动画 + 最终结果

> 交互单元格需要 Jupyter 内核处于运行状态；拖动滑块 / 切换学习率即可实时看到变化。

## 0. 环境准备与中文字体

In [ ]:
import glob, os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib import animation
from mpl_toolkits.mplot3d import Axes3D  # noqa: 注册 3D 投影
import ipywidgets as widgets
from IPython.display import HTML, display

# ---- 注册系统中文字体（mplfonts 提供的 Noto Sans CJK）----
for _pat in [
    "~/.local/lib/python*/site-packages/mplfonts/fonts/*.otf",
    "~/.local/lib/python*/site-packages/mplfonts/fonts/*.ttf",
    "/opt/conda/envs/*/lib/python*/site-packages/mplfonts/fonts/*.otf",
]:
    for _fp in glob.glob(os.path.expanduser(_pat)):
        try:
            font_manager.fontManager.addfont(_fp)
        except Exception:
            pass

_zh = "Noto Sans CJK SC"
plt.rcParams["font.sans-serif"] = [_zh, "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 110

_installed = {f.name for f in font_manager.fontManager.ttflist}
print("matplotlib:", matplotlib.__version__)
print("中文字体可用:", _zh in _installed)
%matplotlib inline

## 1. 生成数据集

真实模型为一次函数 $y = w_{true}\, x + b_{true}$，再叠加一个小的正态噪声：

$$y_i = w_{true}\,x_i + b_{true} + \varepsilon_i,\qquad \varepsilon_i \sim \mathcal{N}(0,\sigma^2)$$

In [ ]:
TRUE_W, TRUE_B = 2.5, 1.0      # 真实参数
NOISE, N       = 0.6, 120      # 噪声标准差、样本量
X_RANGE        = (-3.0, 3.0)   # x 的取值区间

def generate_data(n=N, true_w=TRUE_W, true_b=TRUE_B, noise=NOISE, seed=42):
    rng = np.random.default_rng(seed)
    x = rng.uniform(*X_RANGE, size=n)
    y = true_w * x + true_b + rng.normal(0.0, noise, size=n)
    return x, y

x, y = generate_data()
_xs = np.array([X_RANGE[0] - 0.2, X_RANGE[1] + 0.2])   # 画拟合直线用

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.scatter(x, y, s=24, alpha=0.65, color="steelblue", label="带噪声的观测数据")
ax.plot(_xs, TRUE_W * _xs + TRUE_B, "g--", lw=2, label=f"真实模型  y = {TRUE_W}x + {TRUE_B}")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title(f"数据集：一次函数 + 正态噪声  (n={N}, σ={NOISE})")
ax.legend(); ax.grid(alpha=0.3)
plt.show()

## 2. 损失函数与梯度推导

用**均方误差**（MSE，除以 2 方便求导）作为损失：

$$J(w,b)=\frac{1}{2m}\sum_{i=1}^{m}\big(wx_i+b-y_i\big)^2$$

对 $w$、$b$ 分别求偏导（链式法则）：

$$\frac{\partial J}{\partial w}=\frac{1}{m}\sum_{i=1}^{m}\big(wx_i+b-y_i\big)\,x_i
\qquad\qquad
\frac{\partial J}{\partial b}=\frac{1}{m}\sum_{i=1}^{m}\big(wx_i+b-y_i\big)$$

> 直观理解：梯度就是「所有样本残差的平均」。残差为正（预测偏大）则沿负方向调 $w,b$。

In [ ]:
def compute_loss(w, b, x, y):
    # w, b 可以是标量，也可以是网格矩阵（最后一个维度留给样本）
    err = (np.asarray(w)[..., None] * x + np.asarray(b)[..., None]) - y
    return np.mean(err ** 2, axis=-1) / 2.0

def compute_grad(w, b, x, y):
    # 返回 (dJ/dw, dJ/db)
    err = (w * x + b) - y
    return np.mean(err * x), np.mean(err)

# 随便验证一下：在起点 (0,0) 处的损失与梯度
w0_, b0_ = 0.0, 0.0
g_w, g_b = compute_grad(w0_, b0_, x, y)
print(f"起点 (w,b)=({w0_}, {b0_})")
print(f"  J={compute_loss(w0_, b0_, x, y):.4f}   dJ/dw={g_w:.4f}   dJ/db={g_b:.4f}")
print(f"真值点 (w,b)=({TRUE_W}, {TRUE_B})")
print(f"  J={compute_loss(TRUE_W, TRUE_B, x, y):.4f}")

## 3. 梯度下降迭代

参数沿负梯度方向更新，学习率 $\alpha$ 控制步长：

$$w \leftarrow w - \alpha\,\frac{\partial J}{\partial w}
\qquad
b \leftarrow b - \alpha\,\frac{\partial J}{\partial b}$$

下面把每一轮的 $(w,b)$ 和 loss 都记录下来，方便后面逐步回放。

In [ ]:
def gradient_descent(x, y, lr, epochs, w0=0.0, b0=0.0):
    w, b = float(w0), float(b0)
    hist = {"w": [w], "b": [b], "loss": [float(compute_loss(w, b, x, y))]}
    with np.errstate(over="ignore", invalid="ignore"):
        for _ in range(epochs):
            dw, db = compute_grad(w, b, x, y)
            w -= lr * dw
            b -= lr * db
            hist["w"].append(w)
            hist["b"].append(b)
            hist["loss"].append(float(compute_loss(w, b, x, y)))
    return {k: np.asarray(v) for k, v in hist.items()}

_h = gradient_descent(x, y, lr=0.1, epochs=100)
print(f"lr=0.1, 100 epoch 后:  w={_h['w'][-1]:.4f}, b={_h['b'][-1]:.4f}, loss={_h['loss'][-1]:.4f}")

## 4. 损失函数的三维曲面

在 $(w,b)$ 平面上铺一张网格，计算每一点的 $J(w,b)$，得到一张「碗状」的凸曲面。
梯度下降就是在碗里从某个起点不断走向碗底的过程。

- 🟩 绿色星号：真实参数位置（碗底附近）
- 底面等高线：帮助在 3D 视角下判断位置

In [ ]:
w_grid = np.linspace(-1.5, 5.5, 70)
b_grid = np.linspace(-2.5, 4.5, 70)
W, B = np.meshgrid(w_grid, b_grid)
Z = compute_loss(W, B, x, y)

# z 轴显示上限：以起点损失为参考，裁掉远处过陡的部分，保证主要区域看得清
Z_TOP = float(compute_loss(0.0, 0.0, x, y) * 1.25)

fig = plt.figure(figsize=(9, 6.5))
ax = fig.add_subplot(111, projection="3d")
surf = ax.plot_surface(W, B, Z, cmap="viridis", alpha=0.9, edgecolor="none",
                       rstride=2, cstride=2)
ax.contour(W, B, Z, zdir="z", offset=0, levels=20, cmap="viridis", alpha=0.5)
ax.scatter([TRUE_W], [TRUE_B], [compute_loss(TRUE_W, TRUE_B, x, y)],
           c="lime", s=160, marker="*", edgecolors="k", label="真实最优 (w,b)")
ax.set_xlabel("w"); ax.set_ylabel("b"); ax.set_zlabel("J(w,b)")
ax.set_zlim(0, Z_TOP)
ax.view_init(elev=32, azim=-58)
ax.set_title("损失函数 J(w,b) 的凸曲面")
ax.legend(loc="upper right")
fig.colorbar(surf, shrink=0.55, aspect=14, label="loss")
plt.show()

## 5. 交互演示：逐步观察梯度下降

- **滑块 epoch**：拖到第几轮，就看到那一轮为止的轨迹与拟合效果。
- **下拉选择学习率 lr**：切换后整条轨迹会重新计算。

**左图**：损失曲面 + $(w,b)$ 下降轨迹（红色箭头表示下降方向与步长）
**右图**：同一轮次下，模型对原始数据的拟合直线

两边通过 epoch 一一对应。

In [ ]:
MAX_EPOCH = 100

def _clip_divergent(ws, bs, zs):
    # 学习率过大时参数会爆炸，截断到第一个失效点前，避免图被撑坏
    ok = (np.isfinite(zs) & np.isfinite(ws) & np.isfinite(bs)
          & (np.abs(ws) < 100) & (np.abs(bs) < 100))
    if np.all(ok):
        return ws, bs, zs, False
    k = int(np.argmax(~ok))          # 第一个失效点
    k = max(k, 1)                    # 至少保留起点
    return ws[:k], bs[:k], zs[:k], True

def make_plots(lr, epoch):
    epoch = int(epoch)
    hist = gradient_descent(x, y, lr=lr, epochs=epoch)
    ws, bs, ls = hist["w"], hist["b"], hist["loss"]
    zs = np.array([compute_loss(w, b, x, y) for w, b in zip(ws, bs)])
    ws, bs, zs, diverged = _clip_divergent(ws, bs, zs)
    w_c, b_c, l_c = ws[-1], bs[-1], zs[-1]

    fig = plt.figure(figsize=(14, 6.2))

    # ---------- 左：3D 曲面 + 轨迹 ----------
    ax1 = fig.add_subplot(121, projection="3d")
    ax1.plot_surface(W, B, Z, cmap="viridis", alpha=0.5, edgecolor="none",
                     rstride=3, cstride=3)
    ax1.contour(W, B, Z, zdir="z", offset=0, levels=18, cmap="viridis", alpha=0.4)

    if len(ws) > 1:                  # 箭头：相邻步之间
        step = max(1, len(ws) // 25)
        idx = np.arange(0, len(ws) - 1, step)
        ax1.quiver(ws[idx], bs[idx], zs[idx],
                   ws[idx + 1] - ws[idx],
                   bs[idx + 1] - bs[idx],
                   zs[idx + 1] - zs[idx],
                   color="red", arrow_length_ratio=0.3, lw=1.3)
    ax1.plot(ws, bs, zs, "-o", color="crimson", ms=3, lw=1.6)
    ax1.scatter([ws[0]], [bs[0]], [zs[0]], c="k", s=75, marker="X", label="起点")
    ax1.scatter([w_c], [b_c], [l_c], c="orange", s=140, marker="*",
                edgecolors="k", zorder=6, label=f"当前 (w={w_c:.2f}, b={b_c:.2f})")
    ax1.scatter([TRUE_W], [TRUE_B], [compute_loss(TRUE_W, TRUE_B, x, y)],
                c="lime", s=150, marker="*", edgecolors="k", label="真实最优")

    ax1.set_xlabel("w"); ax1.set_ylabel("b"); ax1.set_zlabel("J(w,b)")
    ax1.set_zlim(0, Z_TOP)
    ax1.view_init(elev=32, azim=-58)
    _t = f"3D 损失曲面 + 下降轨迹  (lr={lr}, epoch={epoch})"
    if diverged:
        _t += "   [发散!]"
    ax1.set_title(_t, fontsize=11)
    ax1.legend(fontsize=8, loc="upper right")

    # ---------- 右：函数拟合 ----------
    ax2 = fig.add_subplot(122)
    ax2.scatter(x, y, s=24, alpha=0.55, color="steelblue", label="观测数据")
    ax2.plot(_xs, TRUE_W * _xs + TRUE_B, "g--", lw=2, label=f"真实 y={TRUE_W}x+{TRUE_B}")
    ax2.plot(_xs, w_c * _xs + b_c, "r-", lw=2.6,
             label=f"当前 y={w_c:.2f}x+{b_c:.2f}")
    ax2.set_xlabel("x"); ax2.set_ylabel("y")
    ax2.set_xlim(_xs[0], _xs[-1])
    ax2.set_ylim(y.min() - 1.5, y.max() + 1.5)
    ax2.set_title(f"函数拟合效果    loss={l_c:.4f}", fontsize=11)
    ax2.legend(fontsize=9, loc="upper left"); ax2.grid(alpha=0.3)

    fig.suptitle(f"epoch {epoch:3d}      w = {w_c:.3f}      b = {b_c:.3f}      J = {l_c:.4f}",
                 fontsize=13, y=0.99)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()


lr_dd = widgets.Dropdown(options=[0.005, 0.01, 0.05, 0.1, 0.2, 0.35, 0.5, 0.7],
                         value=0.1, description="学习率 lr")
ep_sl = widgets.IntSlider(min=0, max=MAX_EPOCH, step=1, value=0,
                          description="epoch", continuous_update=True)
ui  = widgets.HBox([lr_dd, ep_sl])
out = widgets.interactive_output(make_plots, {"lr": lr_dd, "epoch": ep_sl})
display(ui, out)

## 6. 不同学习率的影响

学习率太大会震荡甚至发散，太小则收敛很慢，合适才能又快又稳。

- 左图：损失随 epoch 的下降曲线（对数坐标）
- 右图：$(w,b)$ 平面的下降轨迹（叠在等高线上）

> 对这个问题，理论上当 $\alpha < 2/\lambda_{max}$ 时收敛；本数据集 $\lambda_{max}\approx \mathrm{Var}(x)\approx 3$，临界约 0.66。

In [ ]:
lrs = [0.005, 0.02, 0.1, 0.3, 0.6, 0.8]
EPOCHS_CMP = 60

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))

for lr in lrs:
    h = gradient_descent(x, y, lr=lr, epochs=EPOCHS_CMP)
    loss = h["loss"].copy()
    loss[~np.isfinite(loss)] = np.nan
    axes[0].plot(loss, lw=2, label=f"lr={lr}")
    axes[1].plot(h["w"], h["b"], "-o", ms=3, lw=1.3, label=f"lr={lr}")

axes[0].set_yscale("log")
axes[0].set_ylim(1e-3, 1e3)
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss (log)")
axes[0].set_title("不同学习率的损失下降曲线")
axes[0].grid(alpha=0.3, which="both"); axes[0].legend()

axes[1].contour(W, B, Z, levels=25, cmap="viridis", alpha=0.55)
axes[1].scatter([TRUE_W], [TRUE_B], c="k", marker="*", s=150, zorder=6, label="真实最优")
axes[1].set_xlabel("w"); axes[1].set_ylabel("b")
axes[1].set_xlim(-1.0, 6.0); axes[1].set_ylim(-2.0, 3.0)
axes[1].set_title("(w,b) 平面的下降轨迹（等高线背景）")
axes[1].grid(alpha=0.3); axes[1].legend()

plt.tight_layout(); plt.show()

## 7. 训练全过程动画

把 epoch 0 → 60 连成动画，同时看曲面上的下降和拟合直线的逼近。

In [ ]:
ANIM_LR, ANIM_EPOCHS, ANIM_STRIDE = 0.1, 60, 1

h = gradient_descent(x, y, lr=ANIM_LR, epochs=ANIM_EPOCHS)
ws, bs, ls = h["w"], h["b"], h["loss"]
zs = np.array([compute_loss(w, b, x, y) for w, b in zip(ws, bs)])
frames = list(range(0, len(ws), ANIM_STRIDE))

fig = plt.figure(figsize=(12, 5.6))
axA = fig.add_subplot(121, projection="3d")
axB = fig.add_subplot(122)

def _frame(i):
    axA.clear(); axB.clear()
    axA.plot_surface(W, B, Z, cmap="viridis", alpha=0.5, edgecolor="none",
                     rstride=3, cstride=3)
    axA.contour(W, B, Z, zdir="z", offset=0, levels=15, cmap="viridis", alpha=0.4)
    axA.plot(ws[:i + 1], bs[:i + 1], zs[:i + 1], "-o", color="crimson", ms=3, lw=1.6)
    axA.scatter([ws[i]], [bs[i]], [zs[i]], c="orange", s=120, marker="*", edgecolors="k")
    axA.set_xlabel("w"); axA.set_ylabel("b"); axA.set_zlabel("J(w,b)")
    axA.set_zlim(0, Z_TOP); axA.view_init(elev=32, azim=-58)
    axA.set_title(f"epoch {i}")

    axB.scatter(x, y, s=20, alpha=0.5, color="steelblue")
    axB.plot(_xs, TRUE_W * _xs + TRUE_B, "g--", lw=2, label="真实")
    axB.plot(_xs, ws[i] * _xs + bs[i], "r-", lw=2.6, label="当前拟合")
    axB.set_xlim(_xs[0], _xs[-1]); axB.set_ylim(y.min() - 1.5, y.max() + 1.5)
    axB.set_title(f"loss = {ls[i]:.4f}")
    axB.legend(fontsize=9, loc="upper left"); axB.grid(alpha=0.3)

anim = animation.FuncAnimation(fig, _frame, frames=frames, interval=180, repeat=False)
plt.close(fig)
HTML(anim.to_jshtml())

## 8. 训练完成后的结果

用较大的 epoch 数跑完，对比学到的参数与真实参数。

In [ ]:
FINAL_LR, FINAL_EPOCHS = 0.1, 200
final = gradient_descent(x, y, lr=FINAL_LR, epochs=FINAL_EPOCHS)
w_f, b_f = final["w"][-1], final["b"][-1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))

axes[0].scatter(x, y, s=24, alpha=0.6, color="steelblue", label="观测数据")
axes[0].plot(_xs, TRUE_W * _xs + TRUE_B, "g--", lw=2.2, label=f"真实 y={TRUE_W}x+{TRUE_B}")
axes[0].plot(_xs, w_f * _xs + b_f, "r-", lw=2.6, label=f"拟合 y={w_f:.3f}x+{b_f:.3f}")
axes[0].set_xlabel("x"); axes[0].set_ylabel("y")
axes[0].set_title(f"训练完成：拟合结果  (lr={FINAL_LR}, {FINAL_EPOCHS} epoch)")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(final["loss"], lw=2, color="crimson")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("loss")
axes[1].set_title("损失函数下降过程")
axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

print("真实参数 : w = {:.4f}, b = {:.4f}".format(TRUE_W, TRUE_B))
print("学习参数 : w = {:.4f}, b = {:.4f}".format(w_f, b_f))
print("参数误差 : Δw = {:.4f}, Δb = {:.4f}".format(abs(w_f - TRUE_W), abs(b_f - TRUE_B)))
print("最终 loss: {:.4f}".format(final["loss"][-1]))

---

# 9. 另一种方式：用**贝叶斯估计**求 w, b

前面的梯度下降属于**频率派（Frequentist）**：把 w, b 当成固定但未知的常数，去最大化似然（= 最小化平方误差），最终得到**一个点估计**。

贝叶斯方法换一个视角：**把 w, b 当成随机变量**，先给一个先验分布 $p(\theta)$，再让数据把它更新成**后验分布** $p(\theta\mid\mathcal{D})$。它不只给出估计值，还给出**不确定性**。

> **关键联系**：高斯先验 + 高斯噪声 ⇒ 后验仍是高斯（**共轭**），有闭式解。
> 而且，高斯先验的 MAP 解恰好等价于 **Ridge 回归（L2 正则）**；先验越弱（方差 $\tau_0^2\to\infty$），越退化成前面的最大似然解。

In [ ]:
# 设计矩阵 Φ = [x, 1]，参数向量 θ = [w, b]
Phi = np.column_stack([x, np.ones_like(x)])
theta_true = np.array([TRUE_W, TRUE_B])

# 高斯先验 θ ~ N(mu0, Sigma0)；取弱先验（方差很大）来贴近"无信息"
mu0 = np.array([0.0, 0.0])
tau0 = 10.0
Sigma0 = np.diag([tau0**2, tau0**2])
Sigma0_inv = np.linalg.inv(Sigma0)

print("设计矩阵 Φ shape:", Phi.shape)
print(f"先验: θ ~ N([0,0], {tau0}^2 · I)")

## 9.1 数学推导

**似然**（给定 $\theta$ 和噪声方差 $\sigma^2$）：

$$y \mid \Phi,\theta,\sigma^2 \;\sim\; \mathcal{N}(\Phi\theta,\ \sigma^2 I)$$

**先验**：$\quad\theta \sim \mathcal{N}(\mu_0,\ \Sigma_0)$

高斯 × 高斯仍是高斯，配方（complete the square）后得**后验**仍是高斯：

$$\boxed{\ \theta\mid\mathcal{D},\sigma^2 \;\sim\; \mathcal{N}(\mu_n,\ \Sigma_n)\ }$$

$$\Sigma_n=\Big(\Sigma_0^{-1}+\tfrac{1}{\sigma^2}\Phi^{\top}\Phi\Big)^{-1}
\qquad
\mu_n=\Sigma_n\Big(\Sigma_0^{-1}\mu_0+\tfrac{1}{\sigma^2}\Phi^{\top}y\Big)$$

- $\mu_n$：**后验均值**，用作点估计（也是最小均方误差意义下的最优估计）
- $\Sigma_n$：**后验协方差**，对角线开方就是 $w$、$b$ 的标准差 → 这就是**不确定性**
- 当 $\tau_0\to\infty$（先验无穷弱）时 $\mu_n\to$ 最大似然解（即梯度下降的收敛点）

In [ ]:
# ---- 情形 A：假设噪声方差 σ² 已知，后验有闭式解 ----
sigma = NOISE      # 真实噪声标准差（现实中通常未知，见 9.2 的 Gibbs 采样）

Sigma_post = np.linalg.inv(Sigma0_inv + Phi.T @ Phi / sigma**2)
mu_post    = Sigma_post @ (Sigma0_inv @ mu0 + Phi.T @ y / sigma**2)
std_post   = np.sqrt(np.diag(Sigma_post))

# 频率派最大似然解（正规方程）= 前面梯度下降收敛点
theta_mle = np.linalg.solve(Phi.T @ Phi, Phi.T @ y)

print("           参数估计对比")
print("-" * 44)
print(f"{'真实值':<16}w={TRUE_W:>8.4f}   b={TRUE_B:>8.4f}")
print(f"{'最大似然(梯度下降)':<16}w={theta_mle[0]:>8.4f}   b={theta_mle[1]:>8.4f}")
print(f"{'贝叶斯后验均值':<16}w={mu_post[0]:>8.4f}   b={mu_post[1]:>8.4f}")
print(f"{'后验标准差':<17}w={std_post[0]:>8.4f}   b={std_post[1]:>8.4f}")

In [ ]:
# 在 (w,b) 平面上画出后验概率密度等高线，叠加梯度下降轨迹
wg = np.linspace(mu_post[0] - 4*std_post[0], mu_post[0] + 4*std_post[0], 220)
bg = np.linspace(mu_post[1] - 4*std_post[1], mu_post[1] + 4*std_post[1], 220)
WG, BG = np.meshgrid(wg, bg)
d = np.stack([WG - mu_post[0], BG - mu_post[1]], axis=-1)
Sigma_post_inv = np.linalg.inv(Sigma_post)
dens = np.exp(-0.5 * np.einsum('...i,ij,...j->...', d, Sigma_post_inv, d))

gd = gradient_descent(x, y, lr=0.1, epochs=100)   # 复用前面定义的梯度下降

fig, ax = plt.subplots(figsize=(7.8, 6.2))
cs = ax.contour(WG, BG, dens, levels=10, cmap="Blues")
ax.clabel(cs, inline=True, fontsize=7)
ax.contour(WG, BG, dens, levels=[np.exp(-2.0), np.exp(-0.5)],
           colors=["darkslateblue", "navy"], linewidths=2,
           linestyles=[":", "--"])   # 2σ 与 1σ 置信椭圆
ax.plot(gd["w"], gd["b"], "-o", color="crimson", ms=3, lw=1.5,
        label="梯度下降路径（频率派）")
ax.scatter([mu_post[0]], [mu_post[1]], c="orange", s=190, marker="*",
           edgecolors="k", zorder=6, label="贝叶斯后验均值")
ax.scatter([TRUE_W], [TRUE_B], c="lime", s=170, marker="*",
           edgecolors="k", zorder=6, label="真实参数")
ax.set_xlabel("w"); ax.set_ylabel("b")
ax.set_title("后验分布 p(w,b | 数据) 与梯度下降路径的对比")
ax.legend(); ax.grid(alpha=0.3)
plt.show()

## 9.2 更真实的情形：$\sigma^2$ 也未知 → Gibbs 采样

现实中噪声方差 $\sigma^2$ 也不知道。给 $\sigma^2$ 也配一个先验（逆 Gamma），此时后验**不再有简单闭式解**，但可以用 **Gibbs 采样**：交替从两个「条件分布」抽样，样本来回震荡最终收敛到真实后验。

- $\theta \mid \sigma^2,\mathcal{D} \sim \mathcal{N}(\mu_n,\ \Sigma_n)$ —— 就是上面的闭式高斯
- $\sigma^2 \mid \theta,\mathcal{D} \sim \text{Inv-Gamma}\big(a_0+\tfrac{N}{2},\ b_0+\tfrac{\text{RSS}}{2}\big)$ —— RSS 为残差平方和

反复交替这两步（几十轮后丢弃预热样本），剩下的样本就服从真实后验。用**样本均值估计参数**，用**样本标准差估计不确定性**。

In [ ]:
def gibbs_sampling(Phi, y, mu0, Sigma0_inv, a0=1e-3, b0=1e-3,
                   n_samples=40000, burn_in=10000, seed=0):
    rng = np.random.default_rng(seed)
    N = len(y)
    theta = np.linalg.solve(Phi.T @ Phi, Phi.T @ y)      # 用 MLE 初始化
    sigma2 = float(np.mean((Phi @ theta - y)**2))
    thetas, s2s = [], []
    for t in range(n_samples):
        # 1) θ | σ², D  —— 多元高斯
        V = np.linalg.inv(Sigma0_inv + Phi.T @ Phi / sigma2)
        m = V @ (Sigma0_inv @ mu0 + Phi.T @ y / sigma2)
        L = np.linalg.cholesky(V)
        theta = m + L @ rng.standard_normal(2)
        # 2) σ² | θ, D  —— 逆 Gamma
        rss = float(np.sum((Phi @ theta - y)**2))
        shape = a0 + N / 2
        scale = b0 + rss / 2
        sigma2 = scale / rng.gamma(shape)
        if t >= burn_in:
            thetas.append(theta.copy())
            s2s.append(sigma2)
    return np.array(thetas), np.array(s2s)

samples, sigma2_samples = gibbs_sampling(Phi, y, mu0, Sigma0_inv)
theta_bayes = samples.mean(axis=0)          # 后验均值
theta_bayes_std = samples.std(axis=0)       # 后验标准差
sigma_hat = np.sqrt(sigma2_samples).mean()

print("Gibbs 采样结果：")
print("  后验均值 (w, b) = (%.4f, %.4f)" % tuple(theta_bayes))
print("  后验标准差      = (%.4f, %.4f)" % tuple(theta_bayes_std))
print("  噪声 σ 后验均值 = %.4f   (真实 %.4f)" % (sigma_hat, NOISE))

In [ ]:
# 左：从后验抽出的多条拟合直线 -> 直观体现不确定性
# 右：后验样本 (w,b) 散点 + 真值
fig, axes = plt.subplots(1, 2, figsize=(14, 5.4))

ax = axes[0]
ax.scatter(x, y, s=22, alpha=0.5, color="steelblue", label="观测数据")
idx = np.random.default_rng(1).choice(len(samples), 200, replace=False)
for i in idx:
    ax.plot(_xs, samples[i, 0]*_xs + samples[i, 1], color="orange", alpha=0.05, lw=1)
ax.plot(_xs, theta_bayes[0]*_xs + theta_bayes[1], "r-", lw=2.5, label="后验均值预测")
ax.plot(_xs, TRUE_W*_xs + TRUE_B, "g--", lw=2, label="真实模型")
# 95% 预测区间（含噪声 → 预测方差 = 参数方差 + σ²）
sx = np.linspace(_xs[0], _xs[-1], 100)
Xnew = np.column_stack([sx, np.ones_like(sx)])
mean_pred = Xnew @ theta_bayes
var_pred = sigma2_samples.mean() + np.einsum('ij,jk,ik->i', Xnew, np.cov(samples.T), Xnew)
ax.fill_between(sx, mean_pred - 2*np.sqrt(var_pred), mean_pred + 2*np.sqrt(var_pred),
                color="red", alpha=0.15, label="95% 预测区间")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title("贝叶斯后验预测分布")
ax.legend(fontsize=9); ax.grid(alpha=0.3)

ax = axes[1]
ax.scatter(samples[:, 0], samples[:, 1], s=4, alpha=0.15, color="orange", label="后验样本")
ax.scatter([theta_bayes[0]], [theta_bayes[1]], c="red", s=190, marker="*",
           edgecolors="k", zorder=6, label="后验均值")
ax.scatter([TRUE_W], [TRUE_B], c="lime", s=170, marker="*",
           edgecolors="k", zorder=6, label="真实参数")
ax.set_xlabel("w"); ax.set_ylabel("b")
ax.set_title("参数后验样本 (w, b)")
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

## 9.3 误差对比

三种估计放一起比：

- **参数误差**：$\lVert\hat\theta-\theta_{true}\rVert_2$
- **预测 MSE**：在数据上的均方误差
- 贝叶斯还额外给 **95% 可信区间**，检查真值是否落在区间内

In [ ]:
def pred_mse(theta):
    return float(np.mean((Phi @ theta - y)**2))

def report(name, theta):
    err = np.linalg.norm(theta - theta_true)
    print(f"{name:<24} w={theta[0]:.4f}  b={theta[1]:.4f}   "
          f"|参数误差|={err:.4f}   预测MSE={pred_mse(theta):.4f}")

print("=" * 82)
report("真实参数", theta_true)
report("① 梯度下降（频率派）", theta_mle)
report("② 贝叶斯·已知σ²（闭式）", mu_post)
report("③ 贝叶斯·Gibbs（全贝叶斯）", theta_bayes)
print("=" * 82)

print("\n95% 可信区间检验（真值是否落入）：")
lo, hi = theta_bayes - 1.96*theta_bayes_std, theta_bayes + 1.96*theta_bayes_std
print(f"  Gibbs: w ∈ [{lo[0]:.4f}, {hi[0]:.4f}]  真值 {TRUE_W}  "
      f"{'✓' if lo[0] < TRUE_W < hi[0] else '✗'}")
print(f"         b ∈ [{lo[1]:.4f}, {hi[1]:.4f}]  真值 {TRUE_B}  "
      f"{'✓' if lo[1] < TRUE_B < hi[1] else '✗'}")
lo2, hi2 = mu_post - 1.96*std_post, mu_post + 1.96*std_post
print(f"  闭式 : w ∈ [{lo2[0]:.4f}, {hi2[0]:.4f}]   b ∈ [{lo2[1]:.4f}, {hi2[1]:.4f}]")

## 9.4 小结：两种方法各自的定位

| | 梯度下降（频率派 / MLE） | 贝叶斯估计 |
|---|---|---|
| 参数观 | 固定未知常数 | 随机变量（有分布） |
| 输出 | 一个点估计 $\hat\theta$ | 整个后验分布 $p(\theta\mid\mathcal{D})$ |
| 不确定性 | 需另算（如标准误） | 天然自带（后验协方差 / 可信区间） |
| 与正则的关系 | 无先验 | 高斯先验 = Ridge 的 L2 正则 |
| 计算 | 迭代优化 | 共轭有闭式；一般需 MCMC（如 Gibbs） |

**本例观察到的现象：**

- 弱先验（$\tau_0=10$）下，**贝叶斯后验均值与梯度下降结果几乎完全一致**（都在真值 0.006 误差内）。这说明：**数据足够多时，先验影响会被数据"淹没"，两种学派殊途同归。**
- 贝叶斯的**额外价值**在于：它顺带给出了"我们有多确信"——后验标准差约 $0.03\sim0.06$，可信区间稳稳地包住了真值。
- 若把先验 $\tau_0$ 调小（强先验），后验均值会**向先验均值收缩**——这正是 Ridge 正则抑制过拟合的贝叶斯解释。可以自己改 `tau0` 试试。

## 9.5 用现成的 MCMC 库 `emcee` 做贝叶斯推断

9.2 里我们手写了 Gibbs 采样。实际工作中更常用**通用 MCMC 库**，无需自己推导条件分布。这里用 **`emcee`**——科学计算界最流行的采样器之一，它实现的是 **Affine-Invariant Ensemble Sampling（仿射不变系综采样）**：

- 用一群（ensemble）"walker" 同时在参数空间游走，彼此用**拉伸移动**（stretch move）互相提议新位置
- 相比经典 Metropolis-Hastings，它对参数尺度的缩放不敏感，几乎不用调步长，非常适合这类小维度贝叶斯问题
- 只需要提供**对数概率密度** $\log p(\theta\mid\mathcal{D}) \propto \log p(\mathcal{D}\mid\theta) + \log p(\theta)$

**模型**：3 个未知量 $\theta=(w,\ b,\ \log\sigma)$

$$y_i \sim \mathcal{N}(w x_i + b,\ \sigma^2),\qquad
p(w)=p(b)=\mathcal{N}(0,10^2),\qquad \log\sigma \sim \mathcal{U}(-5,2)$$

In [ ]:
import emcee

theta_true3 = np.array([TRUE_W, TRUE_B, NOISE])

def log_prior(theta):
    w, b, log_s = theta
    if not (-20 < w < 20 and -20 < b < 20 and -5 < log_s < 2):
        return -np.inf
    return -0.5 * ((w/10)**2 + (b/10)**2)      # 高斯先验; log_s 上是均匀先验

def log_likelihood(theta):
    w, b, log_s = theta
    s = np.exp(log_s)
    resid = y - (w * x + b)
    return -0.5*np.sum(resid**2)/s**2 - N*log_s - 0.5*N*np.log(2*np.pi)

def log_probability(theta):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta)

print("对数概率函数定义完成，参数 = (w, b, log σ)")

In [ ]:
# ---- 运行 emcee 系综采样 ----
ndim, nwalkers = 3, 32
nsteps, discard, thin = 6000, 1500, 10

rng_e = np.random.default_rng(7)
p0 = np.array([TRUE_W, TRUE_B, np.log(0.5)]) + 0.1 * rng_e.standard_normal((nwalkers, ndim))

sampler = emcee.EnsembleSampler(nwalkers, ndim, log_probability)
sampler.run_mcmc(p0, nsteps, progress=False)

chain = sampler.get_chain(discard=discard, thin=thin, flat=True)   # 拉平后的后验样本
samples_w, samples_b, samples_s = chain[:, 0], chain[:, 1], np.exp(chain[:, 2])

w_est, b_est = samples_w.mean(), samples_b.mean()
w_std, b_std = samples_w.std(), samples_b.std()

print(f"样本数: {chain.shape[0]:,}   平均接受率: {sampler.acceptance_fraction.mean():.3f}")
print("-" * 56)
print(f"{'':<10}{'w':>12}{'b':>12}{'σ':>12}")
print(f"{'真实值':<10}{TRUE_W:>12.4f}{TRUE_B:>12.4f}{NOISE:>12.4f}")
print(f"{'emcee 估计':<10}{w_est:>12.4f}{b_est:>12.4f}{samples_s.mean():>12.4f}")
print(f"{'后验标准差':<10}{w_std:>12.4f}{b_std:>12.4f}{samples_s.std():>12.4f}")

In [ ]:
# ---- w, b 的二维后验分布图（corner 风格：二维联合 + 两个边缘分布）----
from scipy.stats import gaussian_kde

def kde_credible_levels(ws, bs, levels=(0.68, 0.95), n=160):
    # 用核密度估计得到平滑的联合密度，再取累积概率对应的置信等值线
    kde = gaussian_kde(np.vstack([ws, bs]))
    gx = np.linspace(np.percentile(ws, 0.5), np.percentile(ws, 99.5), n)
    gy = np.linspace(np.percentile(bs, 0.5), np.percentile(bs, 99.5), n)
    GX, GY = np.meshgrid(gx, gy)
    Zg = kde(np.vstack([GX.ravel(), GY.ravel()])).reshape(GX.shape)
    # 求阈值：使密度大于阈值的区域恰好覆盖 levels 指定的概率质量
    zs = np.sort(Zg.ravel())[::-1]
    cum = np.cumsum(zs) / zs.sum()
    thr = [zs[min(np.searchsorted(cum, l), len(zs)-1)] for l in levels]
    return GX, GY, Zg, sorted(thr)

fig = plt.figure(figsize=(8.5, 8.5))
gs = fig.add_gridspec(2, 2, width_ratios=(4, 1), height_ratios=(1, 4),
                      left=0.10, right=0.88, bottom=0.10, top=0.90,
                      wspace=0.06, hspace=0.06)
ax_j  = fig.add_subplot(gs[1, 0])
ax_mw = fig.add_subplot(gs[0, 0], sharex=ax_j)
ax_mb = fig.add_subplot(gs[1, 1], sharey=ax_j)

# 二维联合后验
ax_j.scatter(samples_w, samples_b, s=3, alpha=0.12, color="orange")
GX, GY, Zg, thr = kde_credible_levels(samples_w, samples_b)
ax_j.contour(GX, GY, Zg, levels=thr, colors=["navy", "darkslateblue"],
             linewidths=2.0, linestyles=["-", "--"])
ax_j.scatter([w_est], [b_est], c="red", s=200, marker="*", edgecolors="k",
             zorder=6, label=f"后验均值 ({w_est:.3f}, {b_est:.3f})")
ax_j.scatter([TRUE_W], [TRUE_B], c="lime", s=180, marker="P", edgecolors="k",
             zorder=6, label=f"真实值 ({TRUE_W}, {TRUE_B})")
ax_j.set_xlabel("w"); ax_j.set_ylabel("b")
ax_j.legend(loc="upper right", fontsize=9); ax_j.grid(alpha=0.25)

# 边缘分布（w）
ax_mw.hist(samples_w, bins=60, density=True, color="steelblue", alpha=0.7)
ax_mw.axvline(w_est, color="red", lw=1.6); ax_mw.axvline(TRUE_W, color="green", lw=1.6, ls="--")
ax_mw.set_yticks([]); ax_mw.tick_params(labelbottom=False)
ax_mw.set_title("w, b 的二维后验分布  vs  真实值", fontsize=12)

# 边缘分布（b）
ax_mb.hist(samples_b, bins=60, density=True, orientation="horizontal",
           color="steelblue", alpha=0.7)
ax_mb.axhline(b_est, color="red", lw=1.6); ax_mb.axhline(TRUE_B, color="green", lw=1.6, ls="--")
ax_mb.set_xticks([]); ax_mb.tick_params(labelleft=False)

plt.show()

print(f"95% 可信区间:  w ∈ [{np.percentile(samples_w,2.5):.4f}, {np.percentile(samples_w,97.5):.4f}]")
print(f"               b ∈ [{np.percentile(samples_b,2.5):.4f}, {np.percentile(samples_b,97.5):.4f}]")

In [ ]:
# ---- trace 图：检查 walker 是否收敛（各条链应在同一水平带内充分混合）----
full = sampler.get_chain()          # 未丢弃，形状 (nsteps, nwalkers, ndim)
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

for k, (ax, name) in enumerate(zip(axes, ["w", "b"])):
    ax.plot(full[:, :, k], alpha=0.35, lw=0.6)
    ax.axhline(theta_true3[k], color="green", ls="--", lw=1.5, label="真实值")
    ax.axvline(discard, color="k", ls=":", lw=1.2, label="预热结束")
    ax.set_ylabel(name); ax.grid(alpha=0.3); ax.legend(fontsize=8)
axes[-1].set_xlabel("步数（每个 walker）")
fig.suptitle("emcee 迹线图（trace plot）—— 收敛检验", fontsize=12)
plt.tight_layout(); plt.show()

### 9.6 四种方法总误差对比

把三种频率派/解析方法与 emcee MCMC 放在一起。参数误差用 $\lVert\hat\theta-\theta_{true}\rVert_2$。

In [ ]:
def report2(name, w, b):
    err = np.hypot(w - TRUE_W, b - TRUE_B)
    mse = float(np.mean(((w*x + b) - y)**2))
    print(f"{name:<26} w={w:.4f}  b={b:.4f}   |参数误差|={err:.4f}   预测MSE={mse:.4f}")

print("=" * 84)
report2("真实参数", TRUE_W, TRUE_B)
report2("① 梯度下降（频率派）", theta_mle[0], theta_mle[1])
report2("② 贝叶斯·闭式后验", mu_post[0], mu_post[1])
report2("③ 贝叶斯·手写 Gibbs", theta_bayes[0], theta_bayes[1])
report2("④ 贝叶斯·emcee MCMC", w_est, b_est)
print("=" * 84)
print("\n结论：四种方法参数误差都在同一量级（~0.006），")
print("说明在数据充分、模型正确时，估计方法的选择对点估计影响很小；")
print("MCMC/贝叶斯的主要价值在于同时给出参数的不确定性分布。")